## FORCE catalogue query

FORCE does not use the OpenEO datacube format, based on Geotrellis. Instead, FORCE works with the original SAFE archives of e.g. Sentinel-2.
We would like to avoid doing the query inside the FORCE EOAP, because querying a catalogue is sufficiently general that it could be used by any number of processors.

Therefore, we are designing a process that takes in an area of extent, a time range, and a collection ID and produces a list of s3 urls to the input products to be processed.

In [1]:
import openeo
import json
from pathlib import Path

## Custom process

We have developed a custom process `force_query` that fulfills the above requirements. It is available only on the BC deployment of the backend.

In [2]:
connection_cate = openeo.connect("http://cate:8080").authenticate_oidc()

Authenticated using refresh token.


In [3]:
all_processes = connection_cate.list_processes()
print("FORCE custom processes: ", [p["id"] for p in all_processes if "force" in p["id"]])
print("\n", "-"*40, " force_query ", "-"*40, sep="")

print(json.dumps(next((p for p in all_processes if p["id"] == "force_query")), indent=4))

FORCE custom processes:  ['force_hello_world', 'force_query']

---------------------------------------- force_query ----------------------------------------
{
    "description": "Returns a list of S3 paths",
    "id": "force_query",
    "parameters": [
        {
            "description": "Collection to load",
            "name": "id",
            "optional": false,
            "schema": {
                "type": "string"
            }
        },
        {
            "description": "The date range",
            "name": "temporal_extent",
            "optional": false,
            "schema": {
                "items": {
                    "anyOf": [
                        {
                            "format": "date-time",
                            "subtype": "date-time",
                            "type": "string"
                        },
                        {
                            "format": "date",
                            "subtype": "date",
                      

We can run `force_query` using `datacube_from_process`, even though it does not produce a datacube:

In [4]:
force_query_pg = connection_cate.datacube_from_process(
    "force_query",
    id="SENTINEL2_L1C",
    temporal_extent=["2026-01-01", "2026-01-05"],
    spatial_extent={
        "west": 10.5,
        "south": 44.0,
        "east": 11.5, 
        "north": 45.,
    }
)

res = connection_cate.execute(force_query_pg)
res

['s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQQ_20260102T122232.SAFE/',
 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQP_20260102T122232.SAFE/',
 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPQ_20260102T122232.SAFE/',
 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPP_20260102T122232.SAFE/']

## Passing Query Results to the FORCE level 2 processor

The force level 2 processor `force_level2` can be run as a custom process deployed on the CDSE OpenEO backend or using the CWL document as a UDF.
Our custom query process is not currently deployed there, so the two cannot be combined. To test passing query results to the FORCE processor, we can use a UDF that just returns a list of strings
until we can use the actual query process on CDSE.

In [5]:
connection_cdse = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


### A UDF producing arbitrary strings

The documentation on UDFs focuses mostly on using UDFs on datacubes, using `apply`, `apply_dimension` and similar.
However, using the `apply_udf_data` signature with the `run_udf` process makes it possible to create structured data (e.g. a list) from thin air.

Importantly, the UdfData object **must be modified in-place**. Also, `run_udf`, when not used with the `EOAP-CWL` runtime, **must have the data parameter set as not None**. Therefore, we pass an empty list.

In [6]:
udf_code = f"""
from openeo.udf.udf_data import UdfData
from openeo.udf.structured_data import StructuredData

def apply_udf_data(data: UdfData):
    data.set_structured_data_list([
        StructuredData({res})
    ])
"""
udf = openeo.UDF(udf_code)
udf_process_graph = connection_cdse.datacube_from_process(
    "run_udf",
    data=[],
    udf=udf_code,
    context={},
    runtime="Python",
)

print(udf_process_graph.to_json())

{
  "process_graph": {
    "runudf1": {
      "process_id": "run_udf",
      "arguments": {
        "context": {},
        "data": [],
        "runtime": "Python",
        "udf": "\nfrom openeo.udf.udf_data import UdfData\nfrom openeo.udf.structured_data import StructuredData\n\ndef apply_udf_data(data: UdfData):\n    data.set_structured_data_list([\n        StructuredData(['s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQQ_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQP_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPQ_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPP_20260102T122232.SAFE/'])\n    ])\n"
      },
      "result": true
    }
  }
}


In [7]:
udf_result = udf_process_graph.execute()
udf_result

Preflight process graph validation raised: [FeatureUnsupported] run_udf is not supported in validation mode.


['s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQQ_20260102T122232.SAFE/',
 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQP_20260102T122232.SAFE/',
 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPQ_20260102T122232.SAFE/',
 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPP_20260102T122232.SAFE/']

### Passing Strings as context to FORCE level 2

In [8]:
cwl = Path("/home/hannes/code/apex-force-openeo/material/force-l2.cwl").read_text()

FORCE level 2 expects a number of parameters, which could be passed using the `context` dictionary of `run_udf`. 
The input products are passed with the key `"input"`. However, the following code does not work:
 
```Python
force_pg = connection_cdse.datacube_from_process(
    "run_udf",
    data=None,
    udf=cwl,
    runtime="EOAP-CWL",
    context={"input": udf_process_graph},
)
force_fail_job = connection_cdse.create_job(force_pg)
force_fail_job.start_and_wait() 
```

It fails, because the Python client does not support passing a process graph as one item in the input dictionary. It does however support passing a process graph to replace the entire `context`:

In [9]:
force_pg = connection_cdse.datacube_from_process(
    "run_udf",
    data=None,
    udf=cwl,
    runtime="EOAP-CWL",
    context={"input": udf_process_graph},
)
force_fail_job = connection_cdse.create_job(force_pg)
force_fail_job.start_and_wait() 

Preflight process graph validation failed: Object of type DataCube is not JSON serializable


TypeError: Object of type DataCube is not JSON serializable

In [10]:
force_pg = connection_cdse.datacube_from_process(
    "run_udf",
    data=None,
    udf=cwl,
    runtime="EOAP-CWL",
    context=udf_process_graph,
)
force_pg_json = force_pg.to_json()
print(force_pg_json)

{
  "process_graph": {
    "runudf1": {
      "process_id": "run_udf",
      "arguments": {
        "context": {},
        "data": [],
        "runtime": "Python",
        "udf": "\nfrom openeo.udf.udf_data import UdfData\nfrom openeo.udf.structured_data import StructuredData\n\ndef apply_udf_data(data: UdfData):\n    data.set_structured_data_list([\n        StructuredData(['s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQQ_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQP_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPQ_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPP_20260102T122232.SAFE/'])\n    ])\n"
      }
    },
    "runudf2": {
      "process_id": "run_udf",
      "arguments": {
        "context": {
          "from_node": "runudf1"
        },
        "data"

This does not do exactly what we want, because we have to pass other parameters to FORCE level 2 as well. This may not be a problem when calling force as custom process, instead of using `run_udf`. However, this is a very useful feature and testing via `run_udf` is a good way to develop the application package.

The backend does support passing only part of the context as a sub-graph. It is just the client that does not allow us to do this. Therefore, we can patch the process graph by hand and submit it:

In [11]:
force_pg_json_patched = json.loads(force_pg_json)
force_pg_json_patched["process_graph"]["runudf2"]["arguments"]["context"] = {
    "input": {"from_node": "runudf1"}
    # here could go other parameters
}

In [12]:
force_pg_json_patched

{'process_graph': {'runudf1': {'process_id': 'run_udf',
   'arguments': {'context': {},
    'data': [],
    'runtime': 'Python',
    'udf': "\nfrom openeo.udf.udf_data import UdfData\nfrom openeo.udf.structured_data import StructuredData\n\ndef apply_udf_data(data: UdfData):\n    data.set_structured_data_list([\n        StructuredData(['s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQQ_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQP_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPQ_20260102T122232.SAFE/', 's3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPP_20260102T122232.SAFE/'])\n    ])\n"}},
  'runudf2': {'process_id': 'run_udf',
   'arguments': {'context': {'input': {'from_node': 'runudf1'}},
    'data': None,
    'runtime': 'EOAP-CWL',
    'udf': 'cwlVersion: v1.2\n\n# tested with\n# s

In [13]:
force_job = connection_cdse.create_job(force_pg_json_patched)
force_job.start_and_wait() # This still fails, because accessing the resulting datacube in FORCE format is not yet supported

Preflight process graph validation raised: [FeatureUnsupported] run_udf is not supported in validation mode.


0:00:00 Job 'j-260304101445415695ee010f05868aff': send 'start'
0:00:20 Job 'j-260304101445415695ee010f05868aff': created (progress 0%)
0:00:25 Job 'j-260304101445415695ee010f05868aff': created (progress 0%)
0:00:31 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:00:39 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:00:49 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:01:01 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:01:17 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:01:36 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:02:00 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:02:30 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:03:07 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:03:54 Job 'j-260304101445415695ee010f05868aff': running (progress N/A)
0:04:52 Job 'j-260304101445415695ee010f05868aff': error (progre

JobFailedException: Batch job 'j-260304101445415695ee010f05868aff' didn't finish successfully. Status: error (after 0:04:53).

In [21]:
force_logs = force_job.logs()
[l for l in force_logs if "SAFE" in l.message]

[{'id': '[1772619324501, 575075]',
  'time': '2026-03-04T10:15:24.501Z',
  'level': 'info',
  'message': 'Job spec: {\n "process_graph": {\n  "runudf1": {\n   "process_id": "run_udf",\n   "arguments": {\n    "context": {},\n    "data": [],\n    "runtime": "Python",\n    "udf": "\\nfrom openeo.udf.udf_data import UdfData\\nfrom openeo.udf.structured_data import StructuredData\\n\\ndef apply_udf_data(data: UdfData):\\n    data.set_structured_data_list([\\n        StructuredData([\'s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQQ_20260102T122232.SAFE/\', \'s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TQP_20260102T122232.SAFE/\', \'s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPQ_20260102T122232.SAFE/\', \'s3://eodata/Sentinel-2/MSI/L1C/2026/01/02/S2B_MSIL1C_20260102T101329_N0511_R022_T32TPP_20260102T122232.SAFE/\'])\\n    ])\\n"\n   }\n  },\n  "runudf2": {\n   "process_id": "r

### Conclusion

We can see in the logs, that our inputs are passed as expected.
If we

- Deploy the query process to the operational backend
- Enable passing a process graph as only a part of the context

This design of split query / force level 2 processor will work. The query process could then also be used for other CWL processes that require a query